In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any, Dict, List, Literal, Optional, Tuple

import random
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(DEVICE, torch.cuda.get_device_name(0))
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from from_n3c import *
import re


def normalize_3digit_icd10(code):
    return re.sub(r"[^A-Z0-9]", "", str(code).upper())[:3]


def make_aliases(name, row):
    """Official name + safe surface variants + optional alias/synonym columns."""
    aliases = [str(name).strip()]
    aliases.append(re.sub(r"\[(.*?)\]", r"\1", aliases[0]).replace("(s)", "s"))

    for col in ("aliases", "alias", "synonyms", "synonym"):
        if col in row.index and pd.notna(row[col]):
            aliases.extend(re.split(r"[|;]", str(row[col])))

    return list(dict.fromkeys(
        re.sub(r"\s+", " ", x).strip(" ,;")
        for x in aliases if str(x).strip()
    ))


df_concepts = pd.read_csv('../AdaptivePooling_MLHC/df_icd10.csv')

# Keep the first row for each normalized 3-character ICD-10 code.
concepts, seen_codes = [], set()
for _, row in df_concepts.iterrows():
    code = normalize_3digit_icd10(row.code)
    if not code or code in seen_codes:
        continue
    seen_codes.add(code)
    name = str(row['name']).strip()
    concepts.append({
        'id': code,
        'text': f"{name} (Ancestral category: {icd10_text(code)}, {infer_chapter_from_code(code)})",
        'aliases': make_aliases(name, row),
        'group': str(row.idx_section + 1),
    })

print(f"Concepts: {len(concepts):,}; alias examples: {sum(len(c['aliases']) for c in concepts):,}")

with open("../AdaptivePooling_MLHC/ds_train_chest_trauma_ner.json", "r") as f:
    train_samples = json.load(f)
with open("../AdaptivePooling_MLHC/ds_dev_chest_trauma_ner.json", "r") as f:
    dev_samples = json.load(f)
with open("../AdaptivePooling_MLHC/ds_test_chest_trauma_ner.json", "r") as f:
    val_samples = json.load(f)

for samples in (train_samples, dev_samples, val_samples):
    for sample in samples:
        sample['label'] = 0 if sample['label'] < 3 else 1


In [ ]:
train_samples[0]

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Any, Dict, List, Literal, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1) Dataset + DataLoader for train_samples / val_samples
# ============================================================
class TextLabelDataset(Dataset):
    """
    samples: list of dicts with keys:
      - 'txt'   : str
      - 'label' : int/float/list/np.ndarray/torch.Tensor
    """
    def __init__(self, samples: List[Dict[str, Any]]):
        self.samples = samples
        # basic validation (fail fast)
        for i, s in enumerate(self.samples[:5]):
            if "txt" not in s or "label" not in s:
                raise KeyError(f"Sample at idx {i} missing 'txt' or 'label' keys.")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        s = self.samples[idx]
        return {"txt": s["txt"], "label": s["label"]}


def _infer_label_tensor(labels: List[Any]) -> torch.Tensor:
    """
    Converts batch labels to a tensor:
      - scalar ints/bools -> torch.long  (classification)
      - scalar floats     -> torch.float (regression)
      - vectors           -> torch.float (multi-label / multi-target)
    """
    def to_py(x):
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().tolist() if x.ndim > 0 else x.item()
        if isinstance(x, np.ndarray):
            return x.tolist()
        return x

    labels_py = [to_py(x) for x in labels]
    first = labels_py[0]

    if isinstance(first, (list, tuple)):
        return torch.tensor(labels_py, dtype=torch.float)

    if isinstance(first, (bool, np.bool_)):
        return torch.tensor(labels_py, dtype=torch.long)
    if isinstance(first, (int, np.integer)):
        return torch.tensor(labels_py, dtype=torch.long)
    if isinstance(first, (float, np.floating)):
        return torch.tensor(labels_py, dtype=torch.float)

    t = torch.tensor(labels_py)
    if t.dtype in (torch.int8, torch.int16, torch.int32, torch.int64, torch.uint8):
        return t.long()
    return t.float()


def make_collate_fn(tokenizer, max_length: int = 256, return_text: bool = False):
    """
    Tokenizes texts and batches labels.
    """
    def collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        texts = [b["txt"] for b in batch]
        labels = [b["label"] for b in batch]

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        y = _infer_label_tensor(labels)

        out = {**enc, "labels": y}
        if return_text:
            out["txt"] = texts
        return out

    return collate


def make_dataloader(
    samples: List[Dict[str, Any]],
    tokenizer,
    batch_size: int,
    shuffle: bool,
    max_length: int = 256,
    num_workers: int = 2,
    pin_memory: bool = True,
    return_text: bool = False,
    drop_last: bool = False,
) -> DataLoader:
    ds = TextLabelDataset(samples)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=make_collate_fn(tokenizer, max_length=max_length, return_text=return_text),
        drop_last=drop_last,
    )


# ============================================================
# 2) Concept inputs -> concept_texts + groups
# ============================================================
def concepts_to_texts_and_groups(
    concepts: List[Dict[str, Any]],
    text_key: str = "text",
    group_key: str = "group",
) -> Tuple[List[str], List[List[int]], List[str]]:
    """
    Returns:
      - concept_texts: List[str] length C (REAL concepts only)
      - groups: List[List[int]] group indices over REAL concepts [0..C-1]
      - group_names: parallel list of group identifiers
    """
    # Build concept texts in a fixed order. This order defines concept indices.
    concept_texts: List[str] = []
    for i, c in enumerate(concepts):
        if text_key not in c:
            raise KeyError(f"Concept idx {i} missing '{text_key}'")
        concept_texts.append(str(c[text_key]))

    # Group assignment -> list of concept indices
    group_to_indices: Dict[str, List[int]] = {}
    for i, c in enumerate(concepts):
        if group_key not in c:
            raise KeyError(f"Concept idx {i} missing '{group_key}'")
        g = str(c[group_key])
        group_to_indices.setdefault(g, []).append(i)

    group_names = list(group_to_indices.keys())
    groups = [group_to_indices[g] for g in group_names]
    return concept_texts, groups, group_names


# ============================================================
# 3) Utilities
# ============================================================
def masked_mean(x: torch.Tensor, mask: Optional[torch.Tensor], dim: int) -> torch.Tensor:
    if mask is None:
        return x.mean(dim=dim)
    if mask.dtype != torch.bool:
        mask = mask.bool()
    m = mask
    while m.dim() < x.dim():
        m = m.unsqueeze(-1)
    mf = m.to(x.dtype)
    denom = mf.sum(dim=dim).clamp(min=1.0)
    return (x * mf).sum(dim=dim) / denom


def masked_max(x: torch.Tensor, mask: Optional[torch.Tensor], dim: int, fill_value: float = -1e9) -> torch.Tensor:
    if mask is None:
        return x.max(dim=dim).values
    if mask.dtype != torch.bool:
        mask = mask.bool()
    m = mask
    while m.dim() < x.dim():
        m = m.unsqueeze(-1)
    return x.masked_fill(~m, fill_value).max(dim=dim).values


def sparsemax(logits: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Sparsemax activation: returns sparse probabilities that sum to 1 along dim.
    """
    if dim < 0:
        dim = logits.dim() + dim

    z = logits.transpose(dim, -1)
    orig_shape = z.shape
    z = z.reshape(-1, orig_shape[-1])  # (N, C)

    z_sorted, _ = torch.sort(z, descending=True, dim=-1)
    z_cumsum = z_sorted.cumsum(dim=-1)

    k = torch.arange(1, z.shape[-1] + 1, device=z.device, dtype=z.dtype).view(1, -1)
    cond = 1 + k * z_sorted > z_cumsum
    k_z = cond.sum(dim=-1, keepdim=True).clamp(min=1)

    idx = (k_z - 1).to(torch.long)
    tau = (z_cumsum.gather(dim=-1, index=idx) - 1) / k_z.to(z.dtype)

    p = torch.clamp(z - tau, min=0.0)
    p = p.reshape(orig_shape).transpose(dim, -1)
    return p


# ============================================================
# 4) Build concept embeddings using the SAME encoder as clinical text
# ============================================================
@torch.no_grad()
def build_concept_embeddings(
    concept_texts: List[str],
    tokenizer,
    text_encoder: nn.Module,
    device: torch.device,
    batch_size: int = 32,
    max_length: int = 32,
    pooling: Literal["cls", "mean"] = "cls",
) -> torch.Tensor:
    """
    Encodes concept texts using SAME encoder.
    Returns (C, H) on CPU.
    """
    text_encoder.eval()
    embs = []

    for i in range(0, len(concept_texts), batch_size):
        batch = concept_texts[i : i + batch_size]
        tok = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        out = text_encoder(**tok)
        h = out.last_hidden_state  # (B, L, H)

        if pooling == "cls":
            emb = h[:, 0, :]
        else:
            m = tok["attention_mask"].unsqueeze(-1)
            emb = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)

        embs.append(emb.detach().cpu())

    return torch.cat(embs, dim=0)


# ============================================================
# 5) Ontology-only matcher pretraining (no note concept labels)
# ============================================================
def build_alias_examples(concepts: List[Dict[str, Any]]) -> List[Tuple[str, int]]:
    """Keep aliases that map uniquely to one concept."""
    alias_map: Dict[str, Tuple[str, int]] = {}
    ambiguous = set()

    for concept_idx, concept in enumerate(concepts):
        aliases = concept.get("aliases") or [concept["text"]]
        for alias in aliases:
            alias = re.sub(r"\s+", " ", str(alias)).strip()
            key = alias.casefold()
            if not alias:
                continue
            if key in alias_map and alias_map[key][1] != concept_idx:
                ambiguous.add(key)
            else:
                alias_map[key] = (alias, concept_idx)

    return [value for key, value in alias_map.items() if key not in ambiguous]


def set_matcher_trainable(head: nn.Module, trainable: bool) -> None:
    for module in (head.Wq, head.Wk):
        for parameter in module.parameters():
            parameter.requires_grad = trainable


def pretrain_matcher_from_aliases(
    head: nn.Module,
    text_encoder: nn.Module,
    tokenizer,
    concepts: List[Dict[str, Any]],
    device: torch.device,
    special_token_ids: List[int],
    epochs: int = 3,
    batch_size: int = 32,
    max_length: int = 48,
    lr: float = 1e-4,
    pool_temperature: float = 0.10,
):
    """Contrast each alias against all canonical ICD concept embeddings."""
    examples = build_alias_examples(concepts)
    loader = DataLoader(examples, batch_size=batch_size, shuffle=True, num_workers=0)
    optimizer = torch.optim.AdamW(
        list(head.Wq.parameters()) + list(head.Wk.parameters()), lr=lr
    )
    history = []
    text_encoder.eval()
    set_matcher_trainable(head, True)

    for epoch in range(1, epochs + 1):
        total_loss = total_correct = total_n = 0

        for texts, targets in loader:
            targets = targets.to(device)
            tokenized = tokenizer(
                list(texts), padding=True, truncation=True,
                max_length=max_length, return_tensors="pt",
            ).to(device)

            with torch.no_grad():
                token_embeddings = text_encoder(**tokenized).last_hidden_state

            token_mask = tokenized["attention_mask"].bool()
            for token_id in special_token_ids:
                token_mask &= tokenized["input_ids"] != token_id

            q = F.normalize(head.Wq(token_embeddings), dim=-1)
            k = F.normalize(head.Wk(head.concept_emb), dim=-1)
            token_logits = torch.einsum("blh,ch->blc", q, k) / max(head.temperature, 1e-6)
            token_logits = token_logits.masked_fill(~token_mask.unsqueeze(-1), -1e9)

            valid_length = token_mask.sum(dim=1).clamp(min=1).to(token_logits.dtype)
            alias_logits = pool_temperature * (
                torch.logsumexp(token_logits / pool_temperature, dim=1)
                - valid_length.log().unsqueeze(1)
            )

            loss = F.cross_entropy(alias_logits, targets)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * targets.numel()
            total_correct += (alias_logits.argmax(dim=1) == targets).sum().item()
            total_n += targets.numel()

        row = {
            "epoch": epoch,
            "loss": total_loss / max(total_n, 1),
            "alias_top1": total_correct / max(total_n, 1),
        }
        history.append(row)
        print(
            f"Matcher pretrain {epoch}/{epochs}: "
            f"loss={row['loss']:.4f}, alias_top1={row['alias_top1']:.4f}"
        )

    return pd.DataFrame(history)


# ============================================================
# 5) Model: Mention-aligned AVO with internal NULL concept
# ============================================================
@dataclass
class AVOOutput:
    logits: torch.Tensor
    token_logits: torch.Tensor
    A: torch.Tensor
    V: torch.Tensor
    O: torch.Tensor
    sim: torch.Tensor
    A_pool: Optional[torch.Tensor] = None      # (B, C+1)
    AV_pool: Optional[torch.Tensor] = None     # (B, dv)


class MentionAlignedAVOHead(nn.Module):
    """
    token_logits = A V O (+bias)

    NULL concept handling:
      - A has shape (B,L,C+1) and A[...,0] is the NULL attention.
      - V has shape (C+1,dv) and V[0] is all zeros (NULL contributes no value).
      - Your input concepts are REAL concepts only (C of them).

    Attention is derived from cosine similarity(token, concept) plus a NULL logit.
    """

    def __init__(
        self,
        concept_emb: torch.Tensor,               # (C, H) REAL concepts only
        dv: int,
        num_outputs: int,
        pooling: Literal["max", "mean"] = "mean",
        temperature: float = 0.07,
        gate_margin: float = 0.30,
        gate_tau: float = 0.10,
        top_k: Optional[int] = 8,
        attn_activation: Literal["softmax", "sparsemax"] = "softmax",
        freeze_concepts: bool = True,
        use_bias: bool = True,
        null_bias_init: float = 0.0
    ):
        super().__init__()
        assert concept_emb.ndim == 2
        C, H = concept_emb.shape
        self.C, self.H = C, H
        self.dv, self.num_outputs = dv, num_outputs

        self.pooling = pooling
        self.temperature = float(temperature)
        self.gate_margin = float(gate_margin)
        self.gate_tau = float(gate_tau)
        self.top_k = top_k
        self.attn_activation = attn_activation

        if freeze_concepts:
            self.register_buffer("concept_emb", concept_emb.detach().clone())
        else:
            self.concept_emb = nn.Parameter(concept_emb.detach().clone())

        # Identity initialization reproduces the original cosine matcher before pretraining.
        self.Wq = nn.Linear(H, H, bias=False)
        self.Wk = nn.Linear(H, H, bias=False)
        nn.init.eye_(self.Wq.weight)
        nn.init.eye_(self.Wk.weight)

        self.Wv = nn.Linear(H, dv, bias=False)                 # concept -> value
        self.O = nn.Parameter(torch.randn(dv, num_outputs) * 0.02)  # learnable O
        self.bias = nn.Parameter(torch.zeros(num_outputs)) if use_bias else None
        self.null_bias = nn.Parameter(torch.tensor(float(null_bias_init)))

    def _attn(self, logits_full: torch.Tensor) -> torch.Tensor:
        if self.attn_activation == "softmax":
            return torch.softmax(logits_full, dim=-1)
        if self.attn_activation == "sparsemax":
            return sparsemax(logits_full, dim=-1)
        raise ValueError("attn_activation must be 'softmax' or 'sparsemax'")

    def forward(self, token_embs: torch.Tensor, token_mask: Optional[torch.Tensor] = None) -> AVOOutput:
        B, L, H = token_embs.shape
        if H != self.H:
            raise ValueError(f"token_embs dim {H} must match concept_emb dim {self.H}")

        # Ontology-pretrained token<->concept matcher
        Q = F.normalize(self.Wq(token_embs), p=2, dim=-1)       # (B,L,H)
        K = F.normalize(self.Wk(self.concept_emb), p=2, dim=-1) # (C,H)
        sim = torch.einsum("blh,ch->blc", Q, K)                 # (B,L,C)

        logits_real = sim / max(self.temperature, 1e-6)  # (B,L,C)

        # Optional top-k restriction among real concepts
        if self.top_k is not None and self.top_k < self.C:
            _, idx = torch.topk(logits_real, k=self.top_k, dim=-1)
            keep = torch.zeros_like(logits_real, dtype=torch.bool)
            keep.scatter_(-1, idx, True)
            logits_real = logits_real.masked_fill(~keep, -1e9)

        # NULL logit: dominates when s_max is low
        s_max = sim.max(dim=-1, keepdim=True).values     # (B,L,1)
        null_logit = (self.gate_margin - s_max) / max(self.gate_tau, 1e-6)  # (B,L,1)

        logits_full = torch.cat([null_logit, logits_real], dim=-1)  # (B,L,C+1)
        null_logit = (self.gate_margin - s_max) / max(self.gate_tau, 1e-6)  # (B,L,1)
        null_logit = null_logit + self.null_bias

        # For padding tokens: force attention to NULL only
        if token_mask is not None:
            if token_mask.dtype != torch.bool:
                token_mask = token_mask.bool()
        
            masked = ~token_mask                      # (B, L) True where PAD (or excluded) tokens
            # Set all logits to -inf for masked tokens (across concepts)
            logits_full = logits_full.masked_fill(masked.unsqueeze(-1), -1e9)  # (B, L, C+1)
            # Set NULL logit to 0 for masked tokens so softmax -> [1,0,0,...]
            logits_full[..., 0] = logits_full[..., 0].masked_fill(masked, 0.0)

        # A: (B,L,C+1)
        A = self._attn(logits_full)

        # V: (C+1,dv) with NULL row = 0
        V_real = self.Wv(self.concept_emb)               # (C,dv)
        V_null = V_real.new_zeros(1, self.dv)            # (1,dv)
        V = torch.cat([V_null, V_real], dim=0)           # (C+1,dv)

        # A: (B,L,C+1), token_mask: (B,L)
        if token_mask is not None:
            if token_mask.dtype != torch.bool:
                token_mask = token_mask.bool()
            A_masked = A.masked_fill(~token_mask.unsqueeze(-1), 0.0)   # zero out padded/special tokens
            A_pool = A_masked.max(dim=1).values                         # (B, C+1)
        else:
            A_pool = A.max(dim=1).values                                # (B, C+1)
        
        # (Optional but recommended) prevent NULL from dominating presence-style pooling
        # Comment this out if you WANT NULL to contribute at sample level.
        # A_pool[:, 0] = 0.0
        
        # sample logits = A_pool V O
        AV_pool = torch.einsum("bc,cd->bd", A_pool, V)                  # (B, dv)
        logits = torch.einsum("bd,do->bo", AV_pool, self.O)             # (B, out)
        if self.bias is not None:
            logits = logits + self.bias

        return AVOOutput(
        logits=logits,
        token_logits=logits,
        A=A,
        V=V,
        O=self.O,
        sim=sim,
        A_pool=A_pool,
        AV_pool=AV_pool,
    )


class MentionAlignedAVOModel(nn.Module):
    """
    Wraps a HuggingFace-style encoder that returns .last_hidden_state.
    Masks special tokens from pooling/explanations.
    """

    def __init__(
        self,
        text_encoder: nn.Module,
        head: MentionAlignedAVOHead,
        special_token_ids: Optional[List[int]] = None,
        freeze_text_encoder: bool = True,
    ):
        super().__init__()
        self.text_encoder = text_encoder
        self.head = head
        self.special_token_ids = special_token_ids or []

        if freeze_text_encoder:
            for p in self.text_encoder.parameters():
                p.requires_grad = False

        # Token embeddings are not changed
        for p in self.text_encoder.embeddings.word_embeddings.parameters():
            p.requires_grad = False
        for p in self.text_encoder.embeddings.position_embeddings.parameters():
            p.requires_grad = False
        for p in self.text_encoder.embeddings.token_type_embeddings.parameters():
            p.requires_grad = False
        for p in self.text_encoder.embeddings.LayerNorm.parameters():
            p.requires_grad = False

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: Optional[torch.Tensor] = None,
    ) -> Tuple[AVOOutput, torch.Tensor]:
        enc_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            enc_kwargs["token_type_ids"] = token_type_ids

        enc_out = self.text_encoder(**enc_kwargs)
        token_embs = enc_out.last_hidden_state  # (B,L,H)

        token_mask = attention_mask.bool()
        for tid in self.special_token_ids:
            token_mask = token_mask & (input_ids != tid)

        out = self.head(token_embs=token_embs, token_mask=token_mask)
        return out, token_mask


# ============================================================
# 6) Optional “mention-like” attention regularizers (no concept labels)
# ============================================================
@dataclass
class MentionRegConfig:
    lambda_null_target: float = 0.02
    null_target: float = 0.85
    lambda_entropy: float = 0.02
    eps: float = 1e-8


def mention_regularizers_avo(out: AVOOutput, token_mask: Optional[torch.Tensor], cfg: MentionRegConfig) -> Dict[str, torch.Tensor]:
    """
    Regularizers on A where A[...,0] is NULL.
    """
    A = out.A
    A_null = A[..., 0]     # (B,L)
    A_real = A[..., 1:]    # (B,L,C)

    losses: Dict[str, torch.Tensor] = {}

    # (1) NULL target: encourage most tokens -> NULL
    mean_null = A_null[token_mask].mean() if token_mask is not None else A_null.mean()
    losses["loss_null_target"] = cfg.lambda_null_target * (mean_null - cfg.null_target) ** 2

    # (2) Low entropy for real concept distribution on non-NULL tokens
    denom = (1.0 - A_null).clamp(min=cfg.eps).unsqueeze(-1)
    p_real = (A_real / denom).clamp(min=cfg.eps)  # (B,L,C)
    entropy = -(p_real * p_real.log()).sum(dim=-1)  # (B,L)

    # weight by non-NULL mass (detach so model can't "cheat" by pushing everything to NULL)
    w = (1.0 - A_null).detach()
    if token_mask is not None:
        m = token_mask.float()
        denom_w = (m * w).sum().clamp(min=1.0)
        ent = (entropy * m * w).sum() / denom_w
    else:
        denom_w = w.sum().clamp(min=1.0)
        ent = (entropy * w).sum() / denom_w

    losses["loss_entropy"] = cfg.lambda_entropy * ent
    losses["loss_reg_total"] = losses["loss_null_target"] + losses["loss_entropy"]
    return losses


# ============================================================
# 7) Plain Group Lasso on Beta = V @ O (exclude NULL row!)
# ============================================================
@dataclass
class GroupLassoConfig:
    lambda_group_lasso: float = 1e-3
    use_sqrt_group_size_weight: bool = True
    eps: float = 1e-8


def prepare_group_index_tensors(groups: List[List[int]], device: torch.device) -> List[torch.Tensor]:
    """
    Precompute index tensors (shifted by +1 to skip NULL row in Beta).
    `groups` are REAL concept indices in [0..C-1].
    Returns list of LongTensor indices into Beta rows [1..C].
    """
    group_tensors: List[torch.Tensor] = []
    for g in groups:
        if len(g) == 0:
            continue
        idx = torch.tensor(g, dtype=torch.long, device=device) + 1  # shift for NULL row
        group_tensors.append(idx)
    return group_tensors


def group_lasso_penalty_on_beta(
    beta_with_null: torch.Tensor,          # (C+1, out)
    group_row_indices: List[torch.Tensor], # each is indices into beta rows (already shifted +1)
    cfg: GroupLassoConfig,
) -> torch.Tensor:
    """
    Plain group lasso: sum_g w_g * ||Beta_g||_F.
    Assumes group indices do NOT include NULL row (we shift +1 beforehand).
    """
    if cfg.lambda_group_lasso <= 0.0:
        return beta_with_null.new_zeros(())

    pen = beta_with_null.new_zeros(())
    for idx in group_row_indices:
        bg = beta_with_null.index_select(0, idx)  # (|g|, out)
        frob = torch.sqrt((bg * bg).sum() + cfg.eps)
        if cfg.use_sqrt_group_size_weight:
            w = math.sqrt(float(idx.numel()))
        else:
            w = 1.0
        pen = pen + w * frob

    return cfg.lambda_group_lasso * pen


# ============================================================
# 8) Training helpers (auto loss choice)
# ============================================================
def infer_task_and_outputs(train_samples: List[Dict[str, Any]]) -> Tuple[str, int]:
    """
    Returns (task, num_outputs).
      - 'multiclass': labels are ints -> num_outputs inferred from max label + 1
      - 'regression': labels are floats -> num_outputs = 1
      - 'multilabel': labels are list/array -> num_outputs = len(label)
    """
    y0 = train_samples[0]["label"]

    if isinstance(y0, (list, tuple, np.ndarray, torch.Tensor)) and not (
        isinstance(y0, torch.Tensor) and y0.ndim == 0
    ):
        # multi-label / multi-target
        if isinstance(y0, torch.Tensor):
            k = int(y0.numel())
        elif isinstance(y0, np.ndarray):
            k = int(y0.size)
        else:
            k = len(y0)
        return "multilabel", k

    if isinstance(y0, (float, np.floating)):
        return "regression", 1

    # default classification
    # infer num classes from train labels (safe)
    ys = [int(s["label"]) for s in train_samples]
    num_classes = int(max(ys)) + 1
    return "multiclass", num_classes


def get_loss_fn(task: str):
    if task == "multiclass":
        return nn.CrossEntropyLoss()
    if task == "multilabel":
        return nn.BCEWithLogitsLoss()
    if task == "regression":
        return nn.MSELoss()
    raise ValueError(f"Unknown task: {task}")


# -------------------------
# AUROC / AUPR (no sklearn required)
# -------------------------
def _binary_auc_roc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    AUROC using rank statistic (Mann–Whitney).
    y_true: {0,1}
    """
    y_true = y_true.astype(np.int64)
    y_score = y_score.astype(np.float64)

    pos = y_true == 1
    neg = ~pos
    n_pos = pos.sum()
    n_neg = neg.sum()
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    # rank scores (average ranks for ties)
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(y_score) + 1, dtype=np.float64)

    # handle ties: average ranks
    sorted_scores = y_score[order]
    i = 0
    while i < len(sorted_scores):
        j = i
        while j + 1 < len(sorted_scores) and sorted_scores[j + 1] == sorted_scores[i]:
            j += 1
        if j > i:
            avg_rank = ranks[order[i:j+1]].mean()
            ranks[order[i:j+1]] = avg_rank
        i = j + 1

    sum_ranks_pos = ranks[pos].sum()
    auc = (sum_ranks_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc)


def _binary_aupr_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """
    Average Precision (AUPR) for binary labels.
    """
    y_true = y_true.astype(np.int64)
    y_score = y_score.astype(np.float64)

    n_pos = (y_true == 1).sum()
    if n_pos == 0:
        return float("nan")

    order = np.argsort(-y_score)  # descending
    y_sorted = y_true[order]

    tp = 0
    fp = 0
    ap = 0.0
    for i, yi in enumerate(y_sorted, start=1):
        if yi == 1:
            tp += 1
            ap += tp / i  # precision at this positive
        else:
            fp += 1
    ap = ap / n_pos
    return float(ap)


def compute_auc_aupr(task: str, y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Returns dict with AUROC/AUPR.
    task:
      - 'multiclass': y_true shape (N,), y_prob shape (N, K)
      - 'multilabel': y_true shape (N, K), y_prob shape (N, K)
      - 'binary1': y_true shape (N,), y_prob shape (N,) (prob of positive)
    """
    out = {}

    if task == "binary1":
        out["AUROC"] = _binary_auc_roc(y_true, y_prob)
        out["AUPR"] = _binary_aupr_ap(y_true, y_prob)
        return out

    if task == "multiclass":
        # One-vs-rest macro
        K = y_prob.shape[1]
        aucs, aps = [], []
        for k in range(K):
            yt = (y_true == k).astype(np.int64)
            ys = y_prob[:, k]
            aucs.append(_binary_auc_roc(yt, ys))
            aps.append(_binary_aupr_ap(yt, ys))
        out["AUROC_macro_ovr"] = float(np.nanmean(aucs))
        out["AUPR_macro_ovr"] = float(np.nanmean(aps))
        return out

    if task == "multilabel":
        # Macro over labels
        K = y_true.shape[1]
        aucs, aps = [], []
        for k in range(K):
            yt = y_true[:, k].astype(np.int64)
            ys = y_prob[:, k]
            aucs.append(_binary_auc_roc(yt, ys))
            aps.append(_binary_aupr_ap(yt, ys))
        out["AUROC_macro"] = float(np.nanmean(aucs))
        out["AUPR_macro"] = float(np.nanmean(aps))
        return out

    raise ValueError(f"Unknown metric task: {task}")


# -------------------------
# Concept group reporting
# -------------------------
def summarize_groups(groups: list[list[int]], group_names: list[str], top_n: int = 12) -> str:
    sizes = np.array([len(g) for g in groups], dtype=int)
    order = np.argsort(-sizes)
    lines = []
    lines.append(f"#groups={len(groups)}, #concepts={sizes.sum()} (REAL, excludes NULL)")
    lines.append(f"group size: mean={sizes.mean():.2f}, median={np.median(sizes):.1f}, max={sizes.max()}, min={sizes.min()}")
    lines.append("Top groups by size:")
    for i in order[:top_n]:
        lines.append(f"  - {group_names[i]}: {sizes[i]}")
    return "\n".join(lines)


def active_groups_by_beta(beta_with_null: torch.Tensor,
                          groups: list[list[int]],
                          group_names: list[str],
                          thresh: float = 1e-3,
                          top_n: int = 12) -> str:
    """
    beta_with_null: (C+1, out), row 0 is NULL
    groups: indices in [0..C-1] for REAL concepts
    """
    device = beta_with_null.device
    norms = []
    for g in groups:
        if len(g) == 0:
            norms.append(0.0)
            continue
        idx = torch.tensor(g, device=device, dtype=torch.long) + 1  # shift for NULL
        bg = beta_with_null.index_select(0, idx)  # (|g|, out)
        gn = torch.sqrt((bg * bg).sum()).item()   # Frobenius norm
        norms.append(gn)

    norms_np = np.array(norms, dtype=float)
    active = (norms_np > thresh)
    lines = []
    lines.append(f"Active groups (||Beta_g||_F > {thresh:g}): {active.sum()} / {len(groups)}")
    order = np.argsort(-norms_np)
    lines.append("Top groups by ||Beta_g||_F:")
    for i in order[:top_n]:
        lines.append(f"  - {group_names[i]}: ||Beta_g||_F={norms_np[i]:.4g}  (size={len(groups[i])})")
    return "\n".join(lines)


# -------------------------
# Attention sparsity report
# -------------------------
class AttnSparsityMeter:
    def __init__(self, thresholds=(1e-3, 1e-2, 5e-2), eps=1e-8):
        self.thresholds = thresholds
        self.eps = eps

        self.n_tokens = 0
        self.sum_null = 0.0
        self.sum_entropy = 0.0
        self.sum_eff = 0.0
        self.sum_top1 = 0.0
        self.sum_top5 = 0.0
        self.sum_counts = {t: 0.0 for t in thresholds}

    @torch.no_grad()
    def update(self, A: torch.Tensor, token_mask: torch.Tensor):
        """
        A: (B,L,C+1), token_mask: (B,L) bool
        """
        A_null = A[..., 0]           # (B,L)
        A_real = A[..., 1:]          # (B,L,C)
        m = token_mask

        # consider only valid tokens
        A_null_v = A_null[m]         # (N,)
        A_real_v = A_real[m]         # (N,C)
        N = A_real_v.shape[0]
        if N == 0:
            return

        # normalize real distribution conditional on non-null
        nonnull = (1.0 - A_null_v).clamp(min=self.eps).unsqueeze(-1)  # (N,1)
        p = (A_real_v / nonnull).clamp(min=self.eps)                  # (N,C)

        # entropy and effective number of concepts
        ent = -(p * p.log()).sum(dim=-1)       # (N,)
        eff = torch.exp(ent)                   # (N,)

        # top-k mass (conditional on non-null)
        top1 = p.max(dim=-1).values
        top5 = p.topk(k=min(5, p.shape[-1]), dim=-1).values.sum(dim=-1)

        # count concepts above thresholds (conditional on non-null)
        for t in self.thresholds:
            cnt = (p > t).sum(dim=-1).float().mean().item()
            self.sum_counts[t] += cnt * N

        self.sum_null += A_null_v.sum().item()
        self.sum_entropy += ent.sum().item()
        self.sum_eff += eff.sum().item()
        self.sum_top1 += top1.sum().item()
        self.sum_top5 += top5.sum().item()
        self.n_tokens += N

    def summary(self) -> str:
        if self.n_tokens == 0:
            return "No valid tokens for attention stats."
        n = self.n_tokens
        lines = []
        lines.append(f"Attention sparsity (over non-padding tokens): N_tokens={n}")
        lines.append(f"  mean NULL attention: {self.sum_null / n:.4f}")
        lines.append(f"  mean entropy (real concepts | non-null): {self.sum_entropy / n:.4f}")
        lines.append(f"  mean effective #concepts = exp(entropy): {self.sum_eff / n:.2f}")
        lines.append(f"  mean top1 mass (real | non-null): {self.sum_top1 / n:.4f}")
        lines.append(f"  mean top5 mass (real | non-null): {self.sum_top5 / n:.4f}")
        for t in self.thresholds:
            lines.append(f"  mean #concepts with p_real>{t:g}: {self.sum_counts[t] / n:.2f}")
        return "\n".join(lines)


# -------------------------
# Validation evaluation: AUROC/AUPR + attention + group summaries
# -------------------------
@torch.no_grad()
def evaluate_val(model, val_loader, task: str, device: torch.device):
    model.eval()

    all_probs = []
    all_true = []

    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, token_mask = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=batch.get("token_type_ids", None),
        )
        y = batch["labels"]

        # probs
        if task == "multiclass":
            probs = torch.softmax(out.logits, dim=-1)           # (B,K)
            all_probs.append(probs.detach().cpu().numpy())
            all_true.append(y.detach().cpu().numpy().astype(np.int64))
        elif task == "multilabel":
            probs = torch.sigmoid(out.logits)                   # (B,K)
            all_probs.append(probs.detach().cpu().numpy())
            all_true.append(y.detach().cpu().numpy())
        else:
            # regression has no AUROC/AUPR; still compute attention stats
            pass

    # metrics
    metrics = {}
    if task == "multiclass":
        y_true = np.concatenate(all_true, axis=0)
        y_prob = np.concatenate(all_probs, axis=0)
        # special-case binary-as-2class: also report AUROC/AUPR for class1
        if y_prob.shape[1] == 2:
            metrics.update(compute_auc_aupr("binary1", y_true, y_prob[:, 1]))
        metrics.update(compute_auc_aupr("multiclass", y_true, y_prob))
    elif task == "multilabel":
        y_true = np.concatenate(all_true, axis=0)
        y_prob = np.concatenate(all_probs, axis=0)
        metrics.update(compute_auc_aupr("multilabel", y_true, y_prob))
    else:
        metrics["note"] = "Regression task: AUROC/AUPR not applicable."

    return metrics


# ============================================================
# Annotation-only grounding evaluation; never used in training
# ============================================================
def build_concept_code_index(concepts):
    index = {}
    for i, concept in enumerate(concepts):
        code = normalize_3digit_icd10(concept["id"])
        if code in index:
            raise ValueError(f"Duplicate normalized concept code: {code}")
        index[code] = i
    return index


def annotation_code(annotation):
    if isinstance(annotation, dict):
        raw = annotation.get("code", annotation.get("id"))
    elif isinstance(annotation, (list, tuple)) and len(annotation) >= 2:
        raw = annotation[1]
    else:
        raw = None
    return normalize_3digit_icd10(raw) if raw is not None else ""


def make_grounding_loader(
    samples, tokenizer, concept_to_idx, num_concepts,
    batch_size=4, max_length=512,
):
    def collate(batch):
        texts = [sample["txt"] for sample in batch]
        encoded = tokenizer(
            texts, padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        concept_labels = torch.zeros(len(batch), num_concepts, dtype=torch.bool)
        for i, sample in enumerate(batch):
            for annotation in sample.get("concepts", []):
                idx = concept_to_idx.get(annotation_code(annotation))
                if idx is not None:
                    concept_labels[i, idx] = True
        return {
            **encoded,
            "labels": torch.tensor([sample["label"] for sample in batch]),
            "concept_labels": concept_labels,
        }

    return DataLoader(
        samples, batch_size=batch_size, shuffle=False,
        num_workers=0, collate_fn=collate,
    )


@torch.no_grad()
def evaluate_topk_grounding(
    model, loader, concepts, device,
    ks=(1, 5, 10), use_contribution=False,
):
    """Hit@k = at least one labeled concept occurs among the top-k concepts."""
    model.eval()
    hits = {k: 0 for k in ks}
    n_valid = 0
    max_k = min(max(ks), len(concepts))

    for batch in loader:
        tensors = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        out, _ = model(
            input_ids=tensors["input_ids"],
            attention_mask=tensors["attention_mask"],
            token_type_ids=tensors.get("token_type_ids"),
        )

        scores = out.A_pool
        beta = out.V @ out.O
        if scores.shape[1] == len(concepts) + 1:
            scores, beta = scores[:, 1:], beta[1:]

        if use_contribution:
            if beta.shape[1] != 2:
                raise ValueError("Contribution hit@k expects a two-logit binary model.")
            scores = scores * (beta[:, 1] - beta[:, 0]).unsqueeze(0)

        labels = tensors["concept_labels"].bool()
        valid = labels.any(dim=1)
        if not valid.any():
            continue

        top_indices = scores[valid].topk(max_k, dim=1).indices
        valid_labels = labels[valid]
        n_valid += int(valid.sum())

        for k in ks:
            kk = min(k, max_k)
            hits[k] += int(
                valid_labels.gather(1, top_indices[:, :kk]).any(dim=1).sum()
            )

    prefix = "contributor" if use_contribution else "presence"
    return {
        "n_labeled_notes": n_valid,
        **{f"{prefix}_hit@{k}": hits[k] / max(n_valid, 1) for k in ks},
    }


In [ ]:
from collections import Counter
print("train label counts:", Counter([s["label"] for s in train_samples]))
print("val   label counts:", Counter([s["label"] for s in val_samples]))

In [ ]:
train_samples=train_samples
val_samples=val_samples
concepts=concepts
model_name="cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
batch_size=4
max_length=512
dv=256
pooling="max"
freeze_text_encoder=True      # preserve the ontology-pretrained matcher
freeze_concepts=True         # keeps concepts as anchors
lr=1e-5
num_epochs=2

# Ontology-only matcher pretraining; no note-level concept labels are used.
matcher_pretrain_epochs=3
matcher_pretrain_batch_size=32
matcher_pretrain_lr=1e-4
matcher_pretrain_max_length=48
matcher_pool_temperature=0.10
freeze_matcher_after_pretrain=True
mention_reg_cfg=MentionRegConfig(lambda_null_target=0.02, null_target=0.95, lambda_entropy=0.02)
group_lasso_cfg=GroupLassoConfig(lambda_group_lasso=1e-3)

try:
    from transformers import AutoTokenizer, AutoModel
except Exception as e:
    raise RuntimeError(
        "This script requires `transformers`. Install with: pip install transformers"
    ) from e

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_global_seed(seed: int = 42, deterministic: bool = True) -> None:
    # Python / NumPy
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Make CuDNN / CUDA deterministic (may slow down)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        # Strict mode (will throw if you use non-deterministic ops)
        torch.use_deterministic_algorithms(True)
        # Needed for some CUDA matmul determinism (esp. A100/H100)
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

    # Tokenizers fork warning + determinism friendliness
    os.environ["TOKENIZERS_PARALLELISM"] = "false"


# Call once
set_global_seed(42, deterministic=True)

In [ ]:
class BlackBoxLM(nn.Module):
    def __init__(self, encoder: nn.Module, num_outputs: int):
        super().__init__()
        self.encoder = encoder
        h = encoder.config.hidden_size
        self.head = nn.Linear(h, num_outputs)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**kwargs)
        cls = out.last_hidden_state[:, 0, :]  # (B,H)
        logits = self.head(cls)               # (B,K)
        return logits, out.last_hidden_state[:, 0, :]

tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name).to(device)
num_classes = 2

# Outcome loaders do not contain concept labels.
train_loader = make_dataloader(train_samples, tokenizer, batch_size=batch_size, shuffle=True,  max_length=max_length)
dev_loader   = make_dataloader(dev_samples,   tokenizer, batch_size=batch_size, shuffle=False, max_length=max_length)
val_loader   = make_dataloader(val_samples,   tokenizer, batch_size=batch_size, shuffle=False, max_length=max_length)

# Concept annotations are used only for held-out grounding evaluation.
concept_to_idx = build_concept_code_index(concepts)
dev_grounding_loader = make_grounding_loader(
    dev_samples, tokenizer, concept_to_idx, len(concepts), batch_size, max_length
)
val_grounding_loader = make_grounding_loader(
    val_samples, tokenizer, concept_to_idx, len(concepts), batch_size, max_length
)

blackbox = BlackBoxLM(encoder, num_outputs=num_classes).to(device)
opt1 = torch.optim.AdamW([p for p in blackbox.parameters() if p.requires_grad], lr=1e-5)
loss_fn = nn.CrossEntropyLoss()

blackbox.train()
for epoch in range(1, 2):  # exactly 1 epoch
    total = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k,v in batch.items() if isinstance(v, torch.Tensor)}
        logits, emb = blackbox(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids", None))
        y = batch["labels"].long()
        loss = loss_fn(logits, y)
        opt1.zero_grad(set_to_none=True)
        loss.backward()
        opt1.step()
        total += float(loss.detach().cpu())

In [ ]:
# ---- tokenizer + encoder
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = blackbox.encoder#AutoModel.from_pretrained(model_name).to(device)

# ---- concept texts + group index lists (REAL concepts only)
concept_texts, groups, group_names = concepts_to_texts_and_groups(concepts, text_key="text", group_key="group")
C = len(concept_texts)
print(f"[info] #concepts (REAL): {C}")
print(f"[info] #groups (plain group lasso): {len(groups)} (example group names: {group_names[:5]})")

# ---- build concept embeddings (same encoder space)
concept_emb = build_concept_embeddings(
    concept_texts=concept_texts,
    tokenizer=tokenizer,
    text_encoder=encoder,
    device=device,
    batch_size=4,
    max_length=256,
    pooling="cls",
).to(device)  # (C,H)

# ---- infer task + outputs
task, num_outputs = infer_task_and_outputs(train_samples)
loss_fn = get_loss_fn(task)
print(f"[info] task={task}, num_outputs={num_outputs}")

# ---- model
head = MentionAlignedAVOHead(
    concept_emb=concept_emb,
    dv=dv,
    num_outputs=num_outputs,
    pooling=pooling,
    temperature=0.07,
    gate_margin=0.85,
    gate_tau=0.05,
    null_bias_init=2,
    top_k=min(4, C),                 # safe if C < 8
    attn_activation="sparsemax",       # try "sparsemax" for sparser attention
    freeze_concepts=freeze_concepts,
    use_bias=True,
).to(device)

model = MentionAlignedAVOModel(
    text_encoder=encoder,
    head=head,
    special_token_ids=getattr(tokenizer, "all_special_ids", []),
    freeze_text_encoder=freeze_text_encoder,
).to(device)

# ---- ontology-only matcher pretraining
print("Dev grounding before matcher pretraining:",
      evaluate_topk_grounding(model, dev_grounding_loader, concepts, device))

matcher_history = pretrain_matcher_from_aliases(
    head=model.head,
    text_encoder=model.text_encoder,
    tokenizer=tokenizer,
    concepts=concepts,
    device=device,
    special_token_ids=getattr(tokenizer, "all_special_ids", []),
    epochs=matcher_pretrain_epochs,
    batch_size=matcher_pretrain_batch_size,
    max_length=matcher_pretrain_max_length,
    lr=matcher_pretrain_lr,
    pool_temperature=matcher_pool_temperature,
)

print("Dev grounding after matcher pretraining:",
      evaluate_topk_grounding(model, dev_grounding_loader, concepts, device))
set_matcher_trainable(model.head, not freeze_matcher_after_pretrain)

# ---- group indices for Beta rows (shifted +1 to skip NULL)
group_row_indices = prepare_group_index_tensors(groups, device=device)

# ---- optimizer
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

# ---- simple train/eval loop
for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch in train_loader:
        # move tensors to device
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch.get("token_type_ids", None)
        labels = batch["labels"]

        out, token_mask = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

        # prediction loss
        if task == "multiclass":
            pred_loss = loss_fn(out.logits, labels.long())
        elif task == "multilabel":
            pred_loss = loss_fn(out.logits, labels.float())
        else:
            # regression
            pred_loss = loss_fn(out.logits.squeeze(-1), labels.float())

        # mention-like regs (optional)
        reg_loss = out.logits.new_zeros(())
        if mention_reg_cfg is not None:
            regs = mention_regularizers_avo(out, token_mask, mention_reg_cfg)
            reg_loss = regs["loss_reg_total"]

        # group lasso on Beta = V @ O (exclude NULL via shifted indices)
        gl_loss = out.logits.new_zeros(())
        if group_lasso_cfg is not None and group_lasso_cfg.lambda_group_lasso > 0.0:
            beta = out.V @ out.O  # (C+1, out), beta[0] is NULL row
            gl_loss = group_lasso_penalty_on_beta(beta, group_row_indices, group_lasso_cfg)

        loss = pred_loss + reg_loss + gl_loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.detach().cpu())
        n_batches += 1

        avg_train_loss = total_loss / max(n_batches, 1)
    
    # ---- evaluation (loss only)
    model.eval()
    metrics = evaluate_val(
        model=model,
        val_loader=dev_loader,
        task=task,                      # "multiclass" or "multilabel" (regression handled)
        device=device
    )
        
    grounding = evaluate_topk_grounding(
        model, dev_grounding_loader, concepts, device, ks=(1, 5, 10)
    )
    print(f"---- Development epoch {epoch} ----")
    print({**metrics, **grounding})

print("Final validation/test outcome:",
      evaluate_val(model, val_loader, task, device))
print("Final validation/test presence grounding:",
      evaluate_topk_grounding(model, val_grounding_loader, concepts, device))
print("Final validation/test contributor grounding:",
      evaluate_topk_grounding(
          model, val_grounding_loader, concepts, device, use_contribution=True
      ))
